<table>
  <tr>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
    <td><div align="left"><font size="30">Finding fiducial markers</font></div></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see section 13.6.1

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow.
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'ipympl', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib ipympl

import numpy as np
import math
from spatialmath import SE3, SO3
from spatialmath.base import e2h, h2e
from machinevisiontoolbox import Image, CentralCamera
from spatialmath.base import plot_point
import matplotlib.pyplot as plt
plt.close("all")   # re-running the notebook from the top starts with no figures


***

Fiducial markers are an effective way to mark points in the environment.  The markers carry a numerical identity and their position, or even pose, with respect to the camera can be determined.

This example uses OpenCV function which are available via the Machine Vision Toolbox.  Two common types of fiducual markers are used in robotics and vision: ApriTags (not covered here) and arUco markers.

## ArUco markers

ArUco markers were developed as a simple way to mark things for augmented reality (AR) applications, hence the AR in their name.  They are just high-contrast grids that encode a number as well as position and orientation.

We will load a synthetic image of some arUco markers in the lab.

In [ ]:
scene = Image.Read("lab-scene.png", rgb=False)
scene.disp();
print(scene)

We are attempting to determine the 3D pose of the arUco markers based on 2D image information, so we need to know some parameters of the imaging geometry.  We create a model of a central projection camera, using as many of the parameters as we know:
-  the focal length, in this case 4.25mm
-  the image size
- the principal point, where the optical axis passes through the image plane
- the pixel size, in this case 1.4μm

From the camera model we can derive the intrinsic parameter matrix

In [ ]:
camera = CentralCamera(f=4.25e-3, imagesize=(4032, 3024), pp=(2016, 1512), rho=1.4e-6)
camera.K

There are many different arUco marker families, here we are using `4x4_1000` which is a $4 \times 4$ grid of squares that can encode numbers from 0 to 9999.  We pass in the marker family, the camera intrinsic parameter matrix, and the side length of the marker.  This last parameter is important, because with perspective projection we cannot tell the difference between some large and distance or small and close.  Knowing the size help us estimate distance.

In [ ]:
markers = scene.fiducial("4x4_1000", K=camera.K, side=67e-3);

The return is a list of Marker objects, each has the id of the marker, and the coordinates of four corners which is enough to estimate the orientation of the planar marker in 3D , which is shown above.  We pick element 2 of the marker list  

In [ ]:
marker = markers[2]
marker.id

which is the marker with `id` of 5, the marker being held by the teddy bear.  The corners of this particular fiducial are

In [ ]:
marker.corners

and we can display them on the original image (zoomed in)

In [ ]:
scene.disp()
plot_point(marker.corners, 'bx')
plt.xlim(800, 1200)
plt.ylim(1200, 800)

The pose of the fiducial, with respect to the camera, as an SE(3) matrix is

In [ ]:
marker.pose

Finally, we can render a coordinate frame associated with the pose of each fiducial, into the original image.  The fiducial's z-axis is normal to its plane.

In [ ]:
for marker in markers:
    marker.draw(scene, length=0.10, thick=20)
scene.disp();